version 5

In [ ]:
# Fix numpy/scipy compatibility first
!pip install --upgrade "numpy>=1.26.0,<2.0.0" "scipy>=1.11.0"

# Install core packages
!pip install --no-deps "unsloth[kaggle] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

# Install transformers and audio packages (with dependencies)
!pip install "datasets==2.21.0" "transformers>=4.41.2" jiwer evaluate torchaudio librosa
!pip install -q torch torchaudio
!pip install pyannote.audio safetensors

In [ ]:
# %% [code]
import os
import gc
import re
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
import unicodedata
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# %% [code]
# CONFIGURATION
class Config:
    TEST_AUDIO_DIR = "/kaggle/input/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription/test/audio"
    TRAIN_AUDIO_DIR = "/kaggle/input/competitions/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription/train/audio"
    TRAIN_ANNOTATION_DIR = "/kaggle/input/competitions/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription/train/annotation"
    
    ASR_MODEL = "bengaliAI/tugstugi_bengaliai-asr_whisper-medium"
    
    # VAD Settings — tuned for Bengali long-form speech
    VAD_THRESHOLD = 0.35         # Lower = more sensitive to soft speech
    MIN_SPEECH_DURATION_MS = 250
    MIN_SILENCE_DURATION_MS = 300  # Higher = fewer mid-sentence splits
    SPEECH_PAD_MS = 150           # Pad speech boundaries to avoid clipping
    MAX_SPEECH_DURATION_S = 25    # Match Whisper chunk length for consistency
    GAP_MERGE_S = 1.0             # Merge segments with gaps < 1s
    
config = Config()


# LOAD PYANNOTE SEGMENTATION MODEL (replacing Silero VAD)
print("Loading custom Bengali segmentation model for VAD...")
try:
    from huggingface_hub import hf_hub_download
    from pyannote.audio import Model
    from safetensors.torch import load_file
    import torch.serialization
    from pyannote.audio.core.task import Specifications, Problem, Resolution
    
    torch.serialization.add_safe_globals([Specifications, Problem, Resolution])
    
    # Download custom segmentation model
    safetensors_path = hf_hub_download(
        repo_id="lucius-40/speaker-segmentation-fine-tuned-bn",
        filename="model.safetensors"
    )
    
    # Load base segmentation model architecture
    segmentation_model = Model.from_pretrained(
        "pyannote/segmentation-3.0",
        use_auth_token=False
    )
    
    # Load custom fine-tuned weights
    custom_weights = load_file(safetensors_path)
    custom_weights = {k.replace("model.", "", 1): v for k, v in custom_weights.items()}
    segmentation_model.load_state_dict(custom_weights, strict=False)
    
    segmentation_model = segmentation_model.to(device)
    segmentation_model.eval()
    
    vad_model = segmentation_model
    print("✅ Custom Bengali segmentation model loaded for VAD")
except Exception as e:
    print(f"⚠️ Segmentation model loading failed: {e}")
    vad_model = None


# LOAD ASR MODEL
print("\nLoading Bengali-adapted Whisper model...")

from transformers import pipeline
import torchaudio
import librosa

asr_pipe = pipeline(
    task="automatic-speech-recognition",
    model=config.ASR_MODEL,
    device=0 if device == "cuda" else -1,
    torch_dtype=torch.float16,
    model_kwargs={"attn_implementation": "sdpa"}
)

print("✅ ASR model loaded")


def load_audio(file_path):
    """Load audio → mono float32 numpy @ 16 kHz."""
    try:
        audio, sr = torchaudio.load(file_path)
        if audio.shape[0] > 1:
            audio = torch.mean(audio, dim=0, keepdim=True)
        if sr != 16000:
            audio = torchaudio.transforms.Resample(sr, 16000)(audio)
        return audio.squeeze().numpy(), 16000
    except:
        audio, sr = librosa.load(file_path, sr=16000, mono=True)
        return audio, sr


def get_vad_segments(audio, sr=16000):
    """
    Pyannote segmentation model → merged speech segments.
    Returns list of (start_sample, end_sample) tuples.
    """
    if vad_model is None:
        return [(0, len(audio))]
    
    try:
        from pyannote.core import Segment
        import torch
        
        # Prepare audio in pyannote format
        audio_tensor = torch.from_numpy(audio).float().unsqueeze(0).to(device)
        
        # Run segmentation model
        with torch.no_grad():
            segmentation = vad_model({
                "waveform": audio_tensor,
                "sample_rate": sr
            })
        
        # Convert segmentation output to speech timestamps
        # Pyannote outputs probabilities per frame, threshold to get speech regions
        speech_probs = segmentation.data[:, :, 1]  # Speech probability (class 1)
        
        # Apply threshold
        is_speech = (speech_probs > config.VAD_THRESHOLD).cpu().numpy()[0]
        
        # Convert frames to samples
        # Pyannote uses 10ms frames by default (possibly different, adjust if needed)
        frame_duration = 0.01  # 10ms frames
        samples_per_frame = int(sr * frame_duration)
        
        # Find speech segments
        speech_timestamps = []
        in_speech = False
        start_frame = 0
        
        for i, is_sp in enumerate(is_speech):
            if is_sp and not in_speech:
                start_frame = i
                in_speech = True
            elif not is_sp and in_speech:
                # End of speech segment
                start_sample = int(start_frame * samples_per_frame)
                end_sample = int(i * samples_per_frame)
                
                # Apply minimum duration filter
                duration_ms = (end_sample - start_sample) / sr * 1000
                if duration_ms >= config.MIN_SPEECH_DURATION_MS:
                    speech_timestamps.append({
                        'start': start_sample,
                        'end': end_sample
                    })
                in_speech = False
        
        # Handle if speech continues to the end
        if in_speech:
            start_sample = int(start_frame * samples_per_frame)
            end_sample = len(audio)
            duration_ms = (end_sample - start_sample) / sr * 1000
            if duration_ms >= config.MIN_SPEECH_DURATION_MS:
                speech_timestamps.append({
                    'start': start_sample,
                    'end': end_sample
                })
        
        if not speech_timestamps:
            return [(0, len(audio))]
        
        # Apply speech padding
        pad_samples = int(config.SPEECH_PAD_MS * sr / 1000)
        for ts in speech_timestamps:
            ts['start'] = max(0, ts['start'] - pad_samples)
            ts['end'] = min(len(audio), ts['end'] + pad_samples)
        
        # Merge segments with gaps < GAP_MERGE_S
        segments = []
        current_start = speech_timestamps[0]['start']
        current_end = speech_timestamps[0]['end']
        
        for i in range(1, len(speech_timestamps)):
            ts = speech_timestamps[i]
            gap_s = (ts['start'] - current_end) / sr
            chunk_s = (current_end - current_start) / sr
            
            if gap_s < config.GAP_MERGE_S and chunk_s < config.MAX_SPEECH_DURATION_S:
                # Small gap + chunk not too long → merge
                current_end = ts['end']
            else:
                segments.append((current_start, current_end))
                current_start = ts['start']
                current_end = ts['end']
        
        segments.append((current_start, current_end))
        
        # Split segments exceeding MAX_SPEECH_DURATION_S
        final_segments = []
        for start, end in segments:
            duration = (end - start) / sr
            if duration > config.MAX_SPEECH_DURATION_S:
                num_chunks = int(np.ceil(duration / config.MAX_SPEECH_DURATION_S))
                chunk_samples = (end - start) // num_chunks
                for i in range(num_chunks):
                    cs = start + i * chunk_samples
                    ce = start + (i + 1) * chunk_samples if i < num_chunks - 1 else end
                    final_segments.append((cs, ce))
            else:
                final_segments.append((start, end))
        
        return final_segments
    
    except Exception as e:
        print(f"VAD error: {e}, using full audio")
        import traceback
        traceback.print_exc()
        return [(0, len(audio))]


# POST-PROCESSING
class BengaliPostProcessor:
    def __init__(self):
        self.corrections = {
            'র্া': 'রা',
            '্্': '',
            'ো': 'ো',
            'অ্য': 'এ্য',
            'য়': 'য়',
            
            # Your specific errors from sample output
            'একস্কিজুনে': 'এক্সকিউশনে',
            'বালো': 'ভালো',
            'গল্পোটি': 'গল্পটি',
            'পাব্লিশের্স': 'পাবলিশার্স',
            'প্রাই঵িট': 'প্রাইভেট',
            'লিমিটিট': 'লিমিটেড',
            'সঙ্রক্খিতো': 'সংগ্রহিত',
            'ভিশাচ': 'বিশেষ',
            'বন্তা': 'বিষয়',
            'জশোবন্কের': 'জবাবের',
        }
    
    def remove_repetitions(self, text):
        words = text.split()
        if len(words) < 3:
            return text
        cleaned, i = [], 0
        while i < len(words):
            word = words[i]
            count = 1
            
            # Count consecutive repeats
            j = i + 1
            while j < len(words) and words[j] == word:
                count += 1
                j += 1
            
            # Keep only 1-2 occurrences
            if count > 2:
                cleaned.extend([word] * min(2, count))
                # print(f"Removed {count-2} repetitions of '{word}'")
            else:
                cleaned.extend(words[i:j])
            
            i = j
        
        return ' '.join(cleaned)
    
    def process(self, text):
        if not text:
            return ""
        text = unicodedata.normalize('NFC', text)
        
        # Remove punctuation
        bengali_punct = "।.,;:!?\"'()[]{}—–-–―…॰"
        for punct in bengali_punct:
            text = text.replace(punct, '')
        text = self.remove_repetitions(text)
        for wrong, right in self.corrections.items():
            text = text.replace(wrong, right)
        text = re.sub(r'\s+', ' ', text).strip()
        
        # Final NFC normalization
        text = unicodedata.normalize('NFC', text)
        
        return text

# Initialize post-processor
post_processor = BengaliPostProcessor()

print("\n" + "="*60)
print("PROCESSING TEST FILES WITH CUSTOM SEGMENTATION + BENGALI ASR")
print("="*60)

def transcribe_with_vad(audio_path):
    """
    VAD-guided transcription:
    1. Custom Bengali segmentation model detects speech regions
    2. Each region transcribed with forced Bengali + greedy decoding
    3. Regions > 25s get Whisper's built-in chunking with stride overlap
    """
    try:
        audio, sr = load_audio(audio_path)
        segments = get_vad_segments(audio, sr)
        
        print(f"   VAD: {len(segments)} segments from {len(audio)/sr:.1f}s audio")
        
        segment_transcripts = []
        
        for seg_idx, (start, end) in enumerate(segments):
            segment_audio = audio[start:end]
            duration = len(segment_audio) / sr
            
            # Skip very short segments (< 0.5s — too short for meaningful speech)
            if duration < 0.5:
                continue
            
            # For segments > 25s, use Whisper's built-in chunking
            if duration > 25:
                result = asr_pipe(
                    {"array": segment_audio, "sampling_rate": sr},
                    chunk_length_s=25,
                    stride_length_s=4,
                    batch_size=8,
                    return_timestamps=False,
                    #generate_kwargs={"task": "transcribe", "language": "bn"},
                )
            else:
                result = asr_pipe(
                    {"array": segment_audio, "sampling_rate": sr},
                    return_timestamps=False,
                    #generate_kwargs={"task": "transcribe", "language": "bn"},
                )
            
            text = result.get("text", "").strip()
            if text:
                segment_transcripts.append(text)
        
        raw_text = " ".join(segment_transcripts)
        
        if not raw_text:
            return "", ""
        
        processed = post_processor.process(raw_text)
        
        torch.cuda.empty_cache()
        gc.collect()
        
        return processed, raw_text
        
    except Exception as e:
        print(f"   Error: {e}")
        return "", ""

# Get test files
test_files = []
if os.path.exists(config.TEST_AUDIO_DIR):
    test_files = sorted([f for f in os.listdir(config.TEST_AUDIO_DIR) if f.endswith('.wav')])

if test_files:
    results = []
    
    for i, audio_file in enumerate(tqdm(test_files, desc="Processing")):
        audio_path = os.path.join(config.TEST_AUDIO_DIR, audio_file)
        file_id = os.path.splitext(audio_file)[0]
        
        print(f"\n[{i+1}/{len(test_files)}] {audio_file}")
        
        transcript, raw = transcribe_with_vad(audio_path)
        
        # Empty string fallback (never use wrong Bengali text)
        if not transcript:
            transcript = ""
        
        transcript = unicodedata.normalize('NFC', transcript)
        if i == 0:
            print("\n" + "="*60)
            print("FIRST FILE RESULTS:")
            print("="*60)
            print(f"Raw:\n{raw[:500]}..." if len(raw) > 500 else f"Raw:\n{raw}")
            print(f"\nProcessed:\n{transcript[:500]}..." if len(transcript) > 500 else f"\nProcessed:\n{transcript}")
            print("="*60 + "\n")
        
        results.append({
            'filename': file_id,
            'transcript': transcript
        })
        
        if (i + 1) % 5 == 0:
            pd.DataFrame(results).to_csv("/kaggle/working/checkpoint.csv", index=False)
            print(f"  Checkpoint saved ({i+1}/{len(test_files)})")
    
    submission_df = pd.DataFrame(results)
    submission_df = submission_df[['filename', 'transcript']]
    
    submission_path = "/kaggle/working/submission.csv"
    submission_df.to_csv(submission_path, index=False)
    
    print(f"\n✅ Submission saved: {submission_path}")
    print(f"Processed {len(submission_df)} files")
    print(submission_df.head())
    
else:
    print("No test files found")
    submission_df = pd.DataFrame({
        'filename': [f'test_{i:03d}' for i in range(1, 25)],
        'transcript': [''] * 24
    })
    submission_df.to_csv("/kaggle/working/submission.csv", index=False)

In [ ]:
import pandas as pd
import re
import unicodedata

# Load the dataset
# Adjust the path if the file is not in the same directory, but since the notebook is side-by-side with the csv, relative path works.
input_file = '/kaggle/working/submission.csv'
output_file = '/kaggle/working/FOTR-submission.csv'

try:
    df = pd.read_csv(input_file)
    print(f"Loaded {input_file} with shape {df.shape}")
except FileNotFoundError:
    print(f"File {input_file} not found. Please ensure it is in the same directory.")
    # Fallback to create dummy data for testing the function if file is missing (optional)
    df = pd.DataFrame({'transcript': ["প্রেসিডেন্ট সিরিজের প্রেসিডেন্ট সিরিজের প্রেসিডেন্ট সিরিজের", "হ্যালো হ্যালো হ্যালো", "আমি আমি যাব যাব", "এক দুই তিন এক দুই তিন"]})


def remove_consecutive_repetitions(text):
    if not isinstance(text, str):
        return text
    
    # Normalize to NFC first
    text = unicodedata.normalize('NFC', text)
    
    words = text.split()
    if not words:
        return text

    # We want to remove repetitions of length 1, 2, 3
    # Check for largest n-grams first to capture "A B C A B C" before handling "A A" inside.
    # We will loop until no changes are made to handle nested complications
    
    limit_n = 5
    has_changed = True
    
    while has_changed:
        has_changed = False
        
        # Try n=3, then n=2, then n=1
        for n in range(limit_n, 0, -1):
            i = 0
            new_words = []
            
            # Iterate through the words
            while i < len(words):
                # Current chunk of size n
                chunk = words[i : i+n]
                
                # Check if enough words remain to form a repetition
                # AND if the next chunk matches the current chunk
                if (i + n <= len(words)) and (i + 2*n <= len(words)) and (words[i+n : i+2*n] == chunk):
                    # Found a repetition!
                    new_words.extend(chunk) # Add the first occurrence
                    
                    # Skip the first occurrence (n words)
                    # And skip the repeated occurrence (n words)
                    # But wait, what if it repeats 3 or 4 times?
                    # "A A A" -> "A" "A" (skip) "A" (next loop iteration would fix?)
                    # Let's clean all subsequent identical chunks in this pass
                    
                    i += n # Move past the first chunk
                    
                    while (i + n <= len(words)) and (words[i : i+n] == chunk):
                        i += n # Skip subsequent identical chunks
                    
                    has_changed = True
                    
                    # Note: We added the chunk once, and skipped ALL repetitions.
                    # This effectively reduces "A A A A" to "A".
                    
                else:
                    new_words.append(words[i])
                    i += 1
            
            if has_changed:
                words = new_words
                # Restart the n-loop from the top (n=3) because correcting a repetition might expose a larger pattern or change indices
                break
    
    result = ' '.join(words)
    
    # Final NFC normalization
    return unicodedata.normalize('NFC', result)

# Apply processing
if 'transcript' in df.columns:
    df['transcript'] = df['transcript'].apply(remove_consecutive_repetitions)
    df.to_csv(output_file, index=False)
    print(f"Processing complete. Saved to {output_file}")
    
    # Verify input vs output sample
    sample_text = "প্রেসিডেন্ট সিরিজের প্রেসিডেন্ট সিরিজের প্রেসিডেন্ট সিরিজের"
    print(f"\nTest Input: '{sample_text}'")
    print(f"Test Output: '{remove_consecutive_repetitions(sample_text)}'")

else:
    print("Column 'transcript' not found in dataframe.")